# API로 Claude에 PDF "업로드"하기

[Claude.ai](https://www.claude.ai)의 아주 유용한 기능 중 하나가 PDF 업로드입니다. 이 기능을 노트북에서 흉내 내 보고, 긴 PDF를 요약해 실제로 동작하는지 확인해 보겠습니다.

먼저 Anthropic 클라이언트를 설치하고, 이 노트북 전체에서 사용할 인스턴스를 만듭니다.

In [ ]:
%pip install anthropic

In [24]:
from anthropic import Anthropic
# While PDF support is in beta, you must pass in the correct beta header
client = Anthropic(default_headers={
    "anthropic-beta": "pdfs-2024-09-25"
  }
)
# For now, only claude-sonnet-4-6 supports PDFs
MODEL_NAME = "claude-sonnet-4-6"

`../multimodal/documents` 디렉터리에 이미 PDF가 준비되어 있습니다. 이 PDF 파일을 base64로 인코딩된 바이트로 변환하겠습니다. Claude API의 [PDF 문서 블록](https://docs.claude.com/en/docs/build-with-claude/pdf-support)이 요구하는 형식입니다. 이 방식은 텍스트뿐 아니라 차트나 그래프 같은 시각 요소에도 그대로 적용됩니다.

In [ ]:
import base64


# Start by reading in the PDF and encoding it as base64
file_name = "../multimodal/documents/constitutional-ai-paper.pdf"
with open(file_name, "rb") as pdf_file:
  binary_data = pdf_file.read()
  base64_encoded_data = base64.standard_b64encode(binary_data)
  base64_string = base64_encoded_data.decode("utf-8")


논문을 내려받아 메모리에 올려 뒀으니, 이제 Claude에게 이 문서로 여러 가지 작업을 시켜 볼 수 있습니다. 간단한 질문과 함께 문서를 모델에 전달하겠습니다.

In [32]:
prompt = """
Please do the following:
1. Summarize the abstract at a kindergarten reading level. (In <kindergarten_abstract> tags.)
2. Write the Methods section as a recipe from the Moosewood Cookbook. (In <moosewood_methods> tags.)
3. Compose a short poem epistolizing the results in the style of Homer. (In <homer_results> tags.)
"""
messages = [
    {
        "role": 'user',
        "content": [
            {"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": base64_string}},
            {"type": "text", "text": prompt}
        ]
    }
]

In [21]:
def get_completion(client, messages):
    return client.messages.create(
        model=MODEL_NAME,
        max_tokens=2048,
        messages=messages
    ).content[0].text

In [33]:
completion = get_completion(client, messages)
print(completion)

<kindergarten_abstract>
The scientists wanted to make computer helpers that are nice and don't do bad things. They taught the computer how to check its own work and fix its mistakes without humans having to tell it what's wrong every time. It's like teaching the computer to be its own teacher! They gave the computer some basic rules to follow, like "be kind" and "don't hurt others." Now the computer can answer questions in a helpful way while still being nice and explaining why some things aren't okay to do.
</kindergarten_abstract>

<moosewood_methods>
Constitutional AI Training Stew
A nourishing recipe for teaching computers to be helpful and harmless

Ingredients:
- 1 helpful AI model, pre-trained
- A bundle of constitutional principles
- Several cups of training data
- A dash of human feedback (for helpfulness only)
- Chain-of-thought reasoning, to taste

Method:
1. Begin by gently simmering your pre-trained AI model in a bath of helpful training data until it responds reliably to 